[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/bezier/bezier.ipynb)

# Bézier Curves of Points, Planes and Motors

A rational Bézier curve adds up its control points, each scaled by its own weight and by a Bernstein polynomial of the curve parameter. In the projective geometric algebra of the plane, points can be added and scaled, so that sum is the curve itself, a point at every parameter, with nothing divided. Negative weights and control points at infinity need no special care, and the same sum blends planes and motors. Planes are the elements dual to points; in the plane, where every figure here is drawn, they are lines. This follows [Harquin, Breuils, Biri and Nozick, *Projective Geometric Algebra powers up Bézier Curves*](https://hal.science/hal-05750929) (WSCG 2026).

Two things come with the algebra on top. The conic that a quadratic curve traces is a map from points to planes, built from the control polygon, and the curve's tangent planes are a curve of planes that envelope the same conic through the inverse map.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from math import comb

import numpy as np

from numga import Extensor, NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.animation import save_figure, show_animation
from examples.geometry.bezier import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
# Points at infinity, and the size of one: the norm of its dual.
Direction = ga.gatype(ga.subspace("x y").complement())
direction_metric = Direction.dual() | Direction.dual()                        # [] Scalar <- (Direction, Direction)


def point(coords: np.ndarray) -> Point:
    """Points at unit weight from their x and y coordinates."""
    return (mv("x y", coords) + mv.w).dual()


def heading(coords: np.ndarray) -> Point:
    """Points at infinity in the given directions."""
    return mv("x y", coords).dual()


# Samples along each curve, and every how many samples a tangent plane or a triangle is drawn.
samples = 400
tangent_stride, frame_stride = 25, 40
parameters = np.linspace(0.0, 1.0, samples + 1)
# Cubic curves: control points, and per case the weights; the third case turns the second control
# point into a point at infinity heading up and to the right.
cubic_points = np.array([[0.0, 0.0], [1.0, 2.0], [3.0, 2.0], [4.0, 0.0]])
cubic_heading = np.array([1.0, 1.0])
cubic_weights = np.array([[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, -0.6, 1.0], [1.0, 1.0, 1.0, 1.0]])
# Quadratic curves on one control triangle: the middle weight against the end weights.
quadratic_points = np.array([[-1.0, 0.0], [0.0, 1.5], [1.0, 0.0]])
quadratic_weights = np.array([[1.0, 0.5, 1.0], [1.0, 1.0, 1.0], [1.0, 2.0, 1.0]])
kinds = ("ellipse", "parabola", "hyperbola")
# The middle weight swept from an ellipse through a parabola to a hyperbola and back, the frames of the
# sweep, and the duration of each.
sweep = (0.3, 2.0)
sweep_frames = 48
duration_ms = 80
# Motors: positions and angles of the control placements, and the triangle they carry.
places = np.array([[0.0, 0.0], [1.5, 2.5], [3.0, -1.0], [4.5, 1.5]])
turns = np.array([0.0, 3.0, 6.0, 9.0])
triangle_corners = np.array([[0.3, 0.0], [-0.15, 0.15], [-0.15, -0.15]])
# The windows the figures show.
cubic_box = (-1.5, 5.5, -2.5, 3.5)
conic_box = (-3.0, 3.0, -2.5, 3.0)

## 1. One sum, any weights

The curve is the sum of the control points, each scaled by its weight and its Bernstein polynomial: a point at every parameter, finite wherever its weight is not zero. A negative weight pushes the curve away from its control point. Where the weights cancel, the curve passes through infinity and comes back from the other side, as the dashed curves in the first and last panel do; in the middle panel the weights nearly cancel and the dashed curve swings far out. A control point at infinity, a direction, pulls the curve that way without anchoring it anywhere. Negating both end weights turns the tangents at the ends around, and the curve traces the rest of the same algebraic curve, dashed. The panels show positive weights, a negative third weight, and a second control point at infinity; nothing divides by the weight until the figure is drawn.

In the notation of rational Bézier curves the curve reads $P(t) = \sum_i B^d_i(t)\,\lambda_i P_i \big/ \sum_i B^d_i(t)\,\lambda_i$.

In [ ]:
def bernstein(degree: int, parameters: np.ndarray) -> np.ndarray:
    """The Bernstein polynomials of a degree at each parameter, on a new last axis."""
    index = np.arange(degree + 1)
    binomials = np.array([comb(degree, k) for k in index])
    return binomials * parameters[..., None] ** index * (1 - parameters[..., None]) ** (degree - index)   # [samples, controls]


def blend(controls: Extensor, weights: np.ndarray, parameters: np.ndarray) -> Extensor:
    """The rational Bézier curve of the controls on the last axis: each control weighted by its weight and
    its Bernstein polynomial, summed. Points, planes and motors alike."""
    basis = bernstein(controls.shape[-1] - 1, parameters) * weights[..., None, :]   # [..., samples, controls]
    return (controls[..., None, :] * basis).sum(axis=-1)                      # [..., samples]


def complementary(weights: np.ndarray) -> np.ndarray:
    """The weights that trace the rest of the same curve: both end weights negated, which turns the
    tangents at the ends around."""
    ends = np.isin(np.arange(weights.shape[-1]), [0, weights.shape[-1] - 1])
    return np.where(ends, -weights, weights)


points = point(cubic_points)                                                   # [controls] Point
ideal = stack([points[0], heading(cubic_heading), points[2], points[3]])       # [controls] Point
cubic_controls = stack([points, points, ideal])                                # [cases, controls] Point
cubics = blend(cubic_controls, cubic_weights, parameters)                      # [cases, samples + 1] Point
cubic_complements = blend(cubic_controls, complementary(cubic_weights), parameters)   # [cases, samples + 1] Point

save_figure(render.draw_curves(cubic_controls, cubics, cubic_complements, cubic_box), "bezier_cubics");

## 2. The conic a quadratic curve traces

A quadratic curve lies on a conic, and the conic follows from the control polygon alone: the chord between the ends against the two tangents at the ends, in the proportion `first * last` to `4 * inner**2` of the weights. A plane pairs with a point to a number, so two planes make a conic: a map from points to planes whose form on a point is the product of the two pairings. The form depends on the end weights only through their product, so negating both leaves it unchanged: the curve and its complement lie on one conic, drawn grey under both. Whether the conic misses, touches or crosses the plane at infinity tells the kind of conic, and that is the conic's form on directions: definite for an ellipse, degenerate for a parabola, indefinite for a hyperbola. The middle weight is a half, one and two.

In matrix notation the conic is the symmetric matrix $C = \lambda_0\lambda_2\,\ell_{02}\ell_{02}^\top - 2\lambda_1^2\,(\ell_{01}\ell_{12}^\top + \ell_{12}\ell_{01}^\top)$ on homogeneous point coordinates, and its kind is the sign of the determinant of its upper-left $2 \times 2$ block, the sign of $\lambda_0\lambda_2 - \lambda_1^2$.

In [ ]:
Conic = ga.gatype((Plane, Point))                                               # Plane <- Point


def pair(first: Plane, second: Plane) -> Conic:
    """Two planes as one conic: its form on a point is the product of the two planes' pairings with it."""
    return (first * (second & Point) + second * (first & Point)) / 2           # [...] Plane <- Point


def conic(controls: Point, weights: np.ndarray) -> Conic:
    """The conic a quadratic curve of points traces, from its control polygon: the chord between its ends
    against the two tangents at them, in the proportion the weights set."""
    start, middle, end = controls[..., 0], controls[..., 1], controls[..., 2]
    first, inner, last = weights[..., 0], weights[..., 1], weights[..., 2]
    return first * last * pair(start & end, start & end) - 4 * inner**2 * pair(start & middle, middle & end)   # [...] Plane <- Point


def at_infinity(shape: Conic) -> Scalar:
    """The eigenvalues of the conic's form on directions."""
    return (Direction & shape(Direction)).eigvalsh(direction_metric)            # [..., 2] Scalar


triangle = point(quadratic_points)                                             # [controls] Point
quadratics = blend(triangle, quadratic_weights, parameters)                    # [cases, samples + 1] Point
quadratic_complements = blend(triangle, complementary(quadratic_weights), parameters)   # [cases, samples + 1] Point
shapes = conic(triangle, quadratic_weights)                                    # [cases] Plane <- Point
on_directions = at_infinity(shapes)                                            # [cases, 2] Scalar

for kind, eigenvalues in zip(kinds, render.scalars(on_directions)):
    print(f"{kind:10s} form on directions: {eigenvalues}")
save_figure(render.draw_conics(triangle, quadratics, quadratic_complements, shapes, conic_box), "bezier_conics");

Undivided, each point of the curve is a vector in the three dimensions of x, y and weight, and the point it gives is the whole line through the origin along it. The rays from the origin to the curve sweep a cone, the zero set of the conic's form, and they pierce the unit sphere along the curve normalised; the equator is the plane at infinity. Dividing by the weight follows each ray to the plane of unit weight, where the figures above are drawn. The control points, scaled by their weights, are the grey dots on their own rays; at the smallest middle weight the middle one sits well inside the sphere, and its ray runs on out to it. As the middle weight sweeps from 0.3 to two and back, the cone meets the sphere in two antipodal loops that miss the equator for an ellipse, touch it for a parabola and cross it for a hyperbola; the complement, its end weights negated, runs on the opposite loop.

In [ ]:
# The middle weight, out and back along a cosine of its logarithm, so the sweep loops smoothly.
phase = np.linspace(0, 2 * np.pi, sweep_frames, endpoint=False)
middle = np.exp(np.log(sweep).mean() - np.log(sweep[1] / sweep[0]) / 2 * np.cos(phase))   # [frames]
sweep_weights = np.stack([np.ones_like(middle), middle, np.ones_like(middle)], axis=-1)   # [frames, 3]
weighted = stack([triangle * sweep_weights, triangle * complementary(sweep_weights)], axis=1)   # [frames, 2, controls] Point
swept = blend(triangle, sweep_weights, parameters)                             # [frames, samples + 1] Point
swept_complements = blend(triangle, complementary(sweep_weights), parameters)  # [frames, samples + 1] Point
swept_shapes = conic(triangle, sweep_weights)                                  # [frames] Plane <- Point

show_animation(render.animate_rays(weighted, swept, swept_complements, swept_shapes), "bezier_cone", duration_ms)

## 3. Tangent planes are a curve of planes

The tangent planes of a quadratic curve are themselves a quadratic curve, of planes. Its control planes are the tangent at the start, the chord, and the tangent at the end, with weights `first * inner`, `first * last / 2` and `inner * last`, and planes add and scale like points, so the same `blend` makes it. The inverse of the conic maps planes to points: it is the conic's envelope, and every tangent plane lies on its form, as every point of the curve lies on the conic's.

In matrix notation the envelope is the adjugate of $C$, acting on plane coordinates.

In [ ]:
def tangents(controls: Point, weights: np.ndarray) -> tuple[Plane, np.ndarray]:
    """The control planes and weights of the tangent planes of a quadratic curve of points, themselves a
    quadratic curve of planes: the tangent at the start, the chord, and the tangent at the end."""
    start, middle, end = controls[..., 0], controls[..., 1], controls[..., 2]
    first, inner, last = weights[..., 0], weights[..., 1], weights[..., 2]
    planes = stack([start & middle, start & end, middle & end], axis=-1)        # [..., 3] Plane
    return planes, np.stack([first * inner, first * last / 2, inner * last], axis=-1)


control_planes, plane_weights = tangents(triangle, quadratic_weights)            # [cases, 3] Plane
tangent_planes = blend(control_planes, plane_weights, parameters[::tangent_stride])   # [cases, tangents] Plane
envelopes = shapes.inverse()                                                   # [cases] Point <- Plane

figure = render.draw_conics(triangle, quadratics, quadratic_complements, shapes, conic_box)
save_figure(render.draw_tangents(tangent_planes, figure), "bezier_tangents");

## 4. Curves of motors

Motors add and scale too, so `blend` makes a curve of them; a sum of unit motors is not a unit motor, so the blend is renormalised, as the paper does. De Casteljau's construction gives a curve on the motors themselves instead: at each level it moves from every control motor towards the next along the screw between them, by the parameter, until one motor is left. The figure is the first section's again, with placements for points: on the left the renormalised blend, on the right the curve on the group. The control placements are the filled triangles, joined by the dotted control polygon; the line is the path the curve carries the origin along, and the outlined triangles show the carried triangle along the way. Both curves start and end at the end placements, and they part in between, the more the larger the turns between placements, here three radians each.

In [ ]:
def geodesic(controls: Motor, parameters: np.ndarray) -> Motor:
    """De Casteljau's construction on the motors of the last axis: each level moves along the screw from
    each control towards the next, by the parameter."""
    controls = controls[..., None, :]                                          # [..., 1, controls] Motor
    fractions = parameters[:, None]                                            # [samples, 1]
    while controls.shape[-1] > 1:
        start, end = controls[..., :-1], controls[..., 1:]
        controls = ((end / start).log() * fractions).exp() * start             # [..., samples, level] Motor
    return controls[..., 0]                                                    # [..., samples] Motor


placements = (((mv.xw * places[:, 0] + mv.yw * places[:, 1]) * 0.5).exp()
              * (mv.xy * (-turns / 2)).exp())                                  # [controls] Motor
blended = blend(placements, np.ones(len(turns)), parameters).normalized()      # [samples + 1] Motor
moving = geodesic(placements, parameters)                                      # [samples + 1] Motor
curves = stack([blended, moving])                                              # [curves, samples + 1] Motor
carried = point(triangle_corners)                                              # [corners] Point
origin = mv.w.dual()                                                           # [] Point
centres = placements >> origin                                                 # [controls] Point
paths = curves >> origin                                                       # [curves, samples + 1] Point

save_figure(render.draw_motors(centres, placements[:, None] >> carried, paths,
                               curves[:, ::frame_stride, None] >> carried), "bezier_motors");

In [ ]:
# checks
# Both arcs of each quadratic curve lie on its conic.
assert np.abs(render.scalars(quadratics & shapes[:, None](quadratics))).max() < 1e-10
assert np.abs(render.scalars(quadratic_complements & shapes[:, None](quadratic_complements))).max() < 1e-10
# On directions the conic is definite, degenerate and indefinite: ellipse, parabola, hyperbola.
eigenvalues = render.scalars(on_directions)
assert (eigenvalues[0] > 0).all() and np.abs(eigenvalues[1]).min() < 1e-10 and eigenvalues[2].prod() < 0
# Every tangent plane passes through its point of the curve and lies on the conic's envelope.
assert np.abs(render.scalars(tangent_planes & quadratics[:, ::tangent_stride])).max() < 1e-10
assert np.abs(render.scalars(tangent_planes & envelopes[:, None](tangent_planes))).max() < 1e-10
# Both curves of motors start and end at the end placements.
assert render.scalars((paths[:, [0, -1]] & centres[[0, -1]]).norm()).max() < 1e-10